# Milestone 6: Integrated Borrower Dashboard

## 1. Check the Saved Inputs

The dashboard will bring together:
- Application score and probability of default.
- Historical booking status and proposed application decision.
- Illustrative loan pricing.
- Early warning risk and warning triggers.
- Provisional credit limit increases and sensitivity results.

Application-time decisions and month-23 monitoring results will be
displayed separately so their timing remains clear.

We first inspect the saved files. Missing monitoring results will mean
“not assessed in this demonstration,” rather than “low risk.”

In [2]:
from pathlib import Path

import pandas as pd
from IPython.display import display

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent

input_paths = {
    "Applicants":
        root / "data/raw/sme_applicant_portfolio.csv",

    "Historical booking":
        root / "data/raw/sme_booking_decision.csv",

    "Development application PDs":
        root / "reports/sme_scorecard_v1/development_scored.csv",

    "Validation application scores":
        root / "reports/sme_scorecard_v1/validation_scored.csv",

    "Held-out early warning results":
        root / "reports/sme_ews_v1/test_scored.csv",

    "Limit-increase decisions":
        root / "reports/sme_limit_increase_v1/account_decisions.csv",

    "Provisional limit-increase offers":
        root / "reports/sme_limit_increase_v1/provisional_offers.csv",

    "Limit-increase sensitivity":
        root / "reports/sme_limit_increase_v1/sensitivity_results.csv"
}

inventory = []

for name, path in input_paths.items():
    exists = path.is_file()

    inventory.append({
        "input": name,
        "exists": exists,
        "rows": len(pd.read_csv(path)) if exists else None,
        "path": str(path.relative_to(root))
    })

display(pd.DataFrame(inventory))

# Locate any policy/pricing exports without changing files.
print("\nPossible policy/pricing report files:")

report_files = sorted((root / "reports").rglob("*.csv"))

policy_pricing_files = [
    path for path in report_files
    if any(
        word in path.name.lower()
        for word in ["policy", "pricing", "booked", "comparison"]
    )
]

if policy_pricing_files:
    for path in policy_pricing_files:
        print(path.relative_to(root))
else:
    print("No policy/pricing CSV exports identified yet.")

print("\nInput inventory completed. No files were modified.")

,input,exists,rows,path
0,Applicants,True,15000,data/raw/sme_applicant_portfolio.csv
1,Historical booking,True,15000,data/raw/sme_booking_decision.csv
2,Development application PDs,True,10500,reports/sme_scorecard_v1/development_scored.csv
3,Validation application scores,True,4500,reports/sme_scorecard_v1/validation_scored.csv
4,Held-out early warning results,True,2967,reports/sme_ews_v1/test_scored.csv
5,Limit-increase decisions,True,2967,reports/sme_limit_increase_v1/account_decision...
6,Provisional limit-increase offers,True,2,reports/sme_limit_increase_v1/provisional_offe...
7,Limit-increase sensitivity,True,8,reports/sme_limit_increase_v1/sensitivity_resu...



Possible policy/pricing report files:
reports/sme_lending_policy_v1/booked_pricing.csv
reports/sme_lending_policy_v1/policy_decisions.csv

Input inventory completed. No files were modified.


## 2. Combine Application Scores, Booking, Policy and Pricing

Create one row for each of the 15,000 applicants.

Join records using applicant_id:
- Application PDs: all 15,000 applicants.
- Historical booking status: all 15,000 applicants.
- Saved scores and proposed policy decisions: 4,500 validation applicants.
- Illustrative pricing: 2,974 historically booked validation applicants.

Missing policy or pricing results mean that the applicant was outside
that module's saved evaluation population. They do not mean a decline
or a zero interest rate.

Historical booking and the proposed policy decision remain separate.
Default outcomes are excluded from this dashboard table.

In [3]:
# Load only the application context needed for the dashboard.
applicants = pd.read_csv(
    root / "data/raw/sme_applicant_portfolio.csv",
    usecols=["applicant_id", "industry", "region", "entity_type"],
    dtype={"applicant_id": "string"}
)

booking = pd.read_csv(
    root / "data/raw/sme_booking_decision.csv",
    dtype={"applicant_id": "string"}
)

score_directory = root / "reports/sme_scorecard_v1"
policy_directory = root / "reports/sme_lending_policy_v1"

development_scores = pd.read_csv(
    score_directory / "development_scored.csv",
    dtype={"applicant_id": "string"}
)
validation_scores = pd.read_csv(
    score_directory / "validation_scored.csv",
    dtype={"applicant_id": "string"}
)

development_scores["application_sample"] = "Development"
validation_scores["application_sample"] = "Validation"

application_scores = pd.concat(
    [development_scores, validation_scores],
    ignore_index=True
).drop(columns=["default_12m"], errors="ignore")

policy_decisions = pd.read_csv(
    policy_directory / "policy_decisions.csv",
    dtype={"applicant_id": "string"}
)

pricing = pd.read_csv(
    policy_directory / "booked_pricing.csv",
    dtype={"applicant_id": "string"}
)

sources = {
    "Applicants": applicants,
    "Booking": booking,
    "Application scores": application_scores,
    "Policy decisions": policy_decisions,
    "Pricing": pricing
}

for name, frame in sources.items():
    assert frame["applicant_id"].notna().all(), f"{name}: missing IDs."
    assert frame["applicant_id"].is_unique, f"{name}: duplicate IDs."
    assert set(frame["applicant_id"]).issubset(
        set(applicants["applicant_id"])
    ), f"{name}: unknown applicant IDs."

dashboard_data = (
    applicants
    .merge(
        booking[["applicant_id", "booked"]],
        on="applicant_id", how="left", validate="one_to_one"
    )
    .merge(
        application_scores,
        on="applicant_id", how="left", validate="one_to_one"
    )
    .merge(
        policy_decisions[["applicant_id", "proposed_decision"]],
        on="applicant_id", how="left", validate="one_to_one"
    )
    .merge(
        pricing[["applicant_id", "break_even_rate", "target_rate"]],
        on="applicant_id", how="left", validate="one_to_one"
    )
)

dashboard_data = dashboard_data.rename(columns={
    "predicted_pd": "application_pd",
    "proposed_decision": "application_policy_decision",
    "break_even_rate": "application_break_even_rate",
    "target_rate": "application_target_rate"
})

dashboard_data["historical_booking_status"] = (
    dashboard_data["booked"].map({1: "Booked", 0: "Not booked"})
)

assert len(dashboard_data) == 15_000
assert dashboard_data["application_pd"].between(0, 1).all()
assert dashboard_data["historical_booking_status"].notna().all()
assert dashboard_data["application_policy_decision"].notna().sum() == 4_500
assert dashboard_data["application_target_rate"].notna().sum() == 2_974
assert "default_12m" not in dashboard_data.columns

print(f"Dashboard applicant rows: {len(dashboard_data):,}")
print(f"Application PDs available: {dashboard_data['application_pd'].count():,}")
print(
    "Saved application-policy decisions: "
    f"{dashboard_data['application_policy_decision'].count():,}"
)
print(
    "Illustrative application pricing available: "
    f"{dashboard_data['application_target_rate'].count():,}"
)
print("Application joins passed.")

Dashboard applicant rows: 15,000
Application PDs available: 15,000
Saved application-policy decisions: 4,500
Illustrative application pricing available: 2,974
Application joins passed.


## 3. Add Monitoring and Credit Limit Increase Results

Attach the saved results for the 2,967 held-out booked accounts:
- Early warning probability, review rank and warning triggers.
- Initial screening and limit-increase decisions.
- Provisional offer amounts and profitability.
- Whether each provisional offer passed every sensitivity scenario.

The watchlist contains the 297 highest-risk held-out accounts.

Application PD and deterioration probability remain separate.
Application decisions describe origination; monitoring and limit-increase
assessments describe the review after 24 months of observed behaviour.

Accounts outside the held-out monitoring population are labelled
“Not assessed in this demonstration.”

In [4]:
ews_directory = root / "reports/sme_ews_v1"
limit_directory = root / "reports/sme_limit_increase_v1"

ews_results = pd.read_csv(
    ews_directory / "test_scored.csv",
    dtype={"applicant_id": "string"}
)

watchlist = pd.read_csv(
    ews_directory / "test_watchlist.csv",
    dtype={"applicant_id": "string"}
)

limit_decisions = pd.read_csv(
    limit_directory / "account_decisions.csv",
    dtype={"applicant_id": "string"}
)

offers = pd.read_csv(
    limit_directory / "provisional_offers.csv",
    dtype={"applicant_id": "string"}
)

for name, frame in [
    ("Early warning results", ews_results),
    ("Watchlist", watchlist),
    ("Limit decisions", limit_decisions),
    ("Offers", offers)
]:
    assert frame["applicant_id"].notna().all(), f"{name}: missing IDs."
    assert frame["applicant_id"].is_unique, f"{name}: duplicate IDs."
    assert set(frame["applicant_id"]).issubset(
        set(dashboard_data["applicant_id"])
    ), f"{name}: unknown applicant IDs."

assert set(ews_results["applicant_id"]) == set(
    limit_decisions["applicant_id"]
)
assert set(watchlist["applicant_id"]).issubset(
    set(ews_results["applicant_id"])
)
assert set(offers["applicant_id"]).issubset(
    set(limit_decisions["applicant_id"])
)

monitoring_export = ews_results[[
    "applicant_id",
    "review_rank",
    "predicted_deterioration",
    "warning_reasons",
    "utilization_recent_3m",
    "deposit_decline_pct",
    "max_dpd_recent_3m"
]].copy()

monitoring_export["on_heldout_watchlist"] = (
    monitoring_export["applicant_id"].isin(watchlist["applicant_id"])
)

limit_export = limit_decisions[[
    "applicant_id",
    "current_credit_limit",
    "latest_outstanding_balance",
    "passes_initial_screen",
    "screen_failure_reasons",
    "simulation_decision"
]].rename(columns={
    "simulation_decision": "limit_increase_decision"
})

offer_export = offers[[
    "applicant_id",
    "provisional_increase",
    "provisional_new_limit",
    "incremental_pretax_roe",
    "passes_all_tested_scenarios"
]].rename(columns={
    "provisional_increase": "base_case_offer_increase",
    "provisional_new_limit": "base_case_offer_new_limit"
})

# Rebuild from the application table if this cell is run again.
columns_to_replace = (
    set(monitoring_export.columns)
    | set(limit_export.columns)
    | set(offer_export.columns)
    | {"monitoring_status"}
) - {"applicant_id"}

dashboard_data = dashboard_data.drop(
    columns=list(columns_to_replace),
    errors="ignore"
)

for frame in [monitoring_export, limit_export, offer_export]:
    dashboard_data = dashboard_data.merge(
        frame,
        on="applicant_id",
        how="left",
        validate="one_to_one"
    )

assessed = dashboard_data["predicted_deterioration"].notna()

dashboard_data["monitoring_status"] = (
    "Not assessed in this demonstration"
)
dashboard_data.loc[assessed, "monitoring_status"] = (
    "Assessed — outside top-10% watchlist"
)
dashboard_data.loc[
    dashboard_data["on_heldout_watchlist"].eq(True),
    "monitoring_status"
] = "Prioritized for review — top-10% watchlist"

# Preserve missing numeric values; do not convert missing offers to zero.
dashboard_data["limit_increase_decision"] = (
    dashboard_data["limit_increase_decision"]
    .fillna("Not assessed in this demonstration")
)

assert len(dashboard_data) == 15_000
assert assessed.sum() == 2_967
assert dashboard_data.loc[assessed, "booked"].eq(1).all()
assert dashboard_data["on_heldout_watchlist"].eq(True).sum() == 297
assert dashboard_data["base_case_offer_increase"].notna().sum() == 2

print(f"Integrated applicant rows: {len(dashboard_data):,}")
print(f"Accounts with monitoring results: {assessed.sum():,}")
print(
    "Held-out watchlist accounts: "
    f"{dashboard_data['on_heldout_watchlist'].eq(True).sum():,}"
)
print(
    "Base-case provisional offers: "
    f"{dashboard_data['base_case_offer_increase'].count():,}"
)
print("Monitoring and limit-increase joins passed.")

Integrated applicant rows: 15,000
Accounts with monitoring results: 2,967
Held-out watchlist accounts: 297
Base-case provisional offers: 2
Monitoring and limit-increase joins passed.


## 4. Export the Integrated Dashboard Dataset

Save the combined applicant-level dataset for the dashboard application.

Preserve missing values so the dashboard can distinguish:
- Results available.
- Assessed, but no offer proposed.
- Not assessed in this demonstration.

Check that applicant IDs, probabilities and offer amounts are retained
after saving and reloading.

In [5]:
import numpy as np

dashboard_directory = root / "reports" / "sme_dashboard_v1"
dashboard_directory.mkdir(parents=True, exist_ok=True)

dashboard_path = dashboard_directory / "borrower_dashboard.csv"

# Outcomes must not appear in the decision dashboard.
excluded_outcomes = {
    "default_12m",
    "deterioration_next_6_12mo"
}

assert excluded_outcomes.isdisjoint(dashboard_data.columns)
assert dashboard_data["applicant_id"].is_unique

dashboard_data.to_csv(dashboard_path, index=False)

reloaded_dashboard = pd.read_csv(
    dashboard_path,
    dtype={"applicant_id": "string"}
)

assert reloaded_dashboard["applicant_id"].tolist() == (
    dashboard_data["applicant_id"].tolist()
)

for column in [
    "application_pd",
    "predicted_deterioration",
    "base_case_offer_increase",
    "incremental_pretax_roe"
]:
    assert np.allclose(
        dashboard_data[column].to_numpy(dtype=float),
        reloaded_dashboard[column].to_numpy(dtype=float),
        equal_nan=True
    ), f"Save/reload mismatch: {column}"

assert reloaded_dashboard["monitoring_status"].tolist() == (
    dashboard_data["monitoring_status"].tolist()
)

assert reloaded_dashboard["limit_increase_decision"].tolist() == (
    dashboard_data["limit_increase_decision"].tolist()
)

print("Integrated dataset export and reload checks passed.")
print(f"Applicants saved: {len(reloaded_dashboard):,}")
print(f"Saved: {dashboard_path.relative_to(root)}")

Integrated dataset export and reload checks passed.
Applicants saved: 15,000
Saved: reports/sme_dashboard_v1/borrower_dashboard.csv


## 5. Complete Application Results for the Demonstration

Use the saved score scaling and exact policy thresholds to populate the
application view for all applicants.

Calculate illustrative pricing for all historically booked applicants
using the original pricing assumptions.

These additional results are demonstration outputs. Independent application
evaluation remains restricted to the original 4,500 validation applicants.
No models are retrained and no thresholds are changed.

In [6]:
import json
import joblib
import numpy as np

bundle = joblib.load(
    root / "models/sme_scorecard_v1/scorecard_bundle.joblib"
)

config = json.loads(
    (
        root / "reports/sme_lending_policy_v1/policy_config.json"
    ).read_text(encoding="utf-8")
)

pd_values = dashboard_data["application_pd"]
assert pd_values.between(0, 1, inclusive="neither").all()

# Reverse the PD-to-logit conversion using the saved score scaling.
application_logit = np.log(pd_values) - np.log1p(-pd_values)

complete_exact_score = (
    bundle["score_offset"]
    - bundle["score_factor"] * application_logit
)
complete_display_score = np.rint(complete_exact_score).astype(int)

# Check against the previously exported scores before filling gaps.
existing_scores = dashboard_data["display_score"].notna()

assert np.array_equal(
    dashboard_data.loc[existing_scores, "display_score"].astype(int),
    complete_display_score.loc[existing_scores]
)

complete_decision = pd.Series(
    np.select(
        [
            pd_values.le(config["approve_cutoff"]),
            pd_values.ge(config["decline_cutoff"])
        ],
        ["Auto-approve", "Auto-decline"],
        default="Manual review"
    ),
    index=dashboard_data.index
)

existing_decisions = (
    dashboard_data["application_policy_decision"].notna()
)

assert (
    dashboard_data.loc[
        existing_decisions, "application_policy_decision"
    ] == complete_decision.loc[existing_decisions]
).all()

# Apply the original illustrative pricing assumptions.
pricing_assumptions = config["pricing_assumptions"]
booked_mask = dashboard_data["booked"].eq(1)

complete_break_even = (
    pricing_assumptions["funding_cost_rate"]
    + pricing_assumptions["operating_cost_rate"]
    + pd_values * pricing_assumptions["lgd"]
).where(booked_mask)

complete_target = (
    complete_break_even + pricing_assumptions["target_margin"]
)

existing_prices = dashboard_data["application_target_rate"].notna()

assert np.allclose(
    dashboard_data.loc[existing_prices, "application_target_rate"],
    complete_target.loc[existing_prices]
)

dashboard_data["exact_score"] = complete_exact_score
dashboard_data["display_score"] = complete_display_score
dashboard_data["application_policy_decision"] = complete_decision
dashboard_data["application_break_even_rate"] = complete_break_even
dashboard_data["application_target_rate"] = complete_target

dashboard_data["application_evaluation_scope"] = np.where(
    dashboard_data["application_sample"].eq("Validation"),
    "Original held-out application evaluation",
    "Development account — demonstration output"
)

dashboard_data.to_csv(dashboard_path, index=False)

saved = pd.read_csv(dashboard_path)
assert len(saved) == 15_000
assert saved["display_score"].notna().all()
assert saved["application_policy_decision"].notna().all()
assert (
    saved["application_target_rate"].notna().sum()
    == booked_mask.sum()
)

print("Original exported scores, decisions and pricing match.")
print(f"Applicants with complete scores and decisions: {len(saved):,}")
print(f"Historically booked accounts with pricing: {booked_mask.sum():,}")
print("Updated dashboard dataset saved.")

Original exported scores, decisions and pricing match.
Applicants with complete scores and decisions: 15,000
Historically booked accounts with pricing: 9,887
Updated dashboard dataset saved.
